# Sentiment Analysis

Machine learning sentiment analysis works by teaching a computer to recognize emotions on its own, instead of giving it a dictionary. 

First, I show the computer thousands of example comments that humans have already labeled as "positive" or "negative." The computer looks at these examples and finds hidden patterns, like which words or phrases usually appear together in a good review versus a bad review. 

Over time, it builds its own mathematical rules for understanding context. Once the training is done, then I can give the model brand new comments it has never seen before, and it will accurately guess the sentiment based on the patterns it learned.

<img src="istockphoto-1435905195-612x612.jpg" width="600" alt="Sentiment Analysis Chart">

In [36]:
import csv

X_txt_all = []
y_all = []

with open(r'C:\Users\151495\Desktop\Sentiment Analysis\Sentiment Dataset.csv', encoding='latin-1') as my_file:
    reader = csv.reader(my_file)
    header = next(reader) 
    for row in reader:
        if len(row) < 3:
            continue
        y_all.append(row[2])   
        X_txt_all.append(row[1]) 


total_rows = len(X_txt_all)
split_index = int(total_rows * 0.80)  

# First assigns 80% into 'X_txt_train' then assigns the rest 20% into 'y_train'
X_txt_train = X_txt_all[:split_index]
y_train = y_all[:split_index]

X_txt_test = X_txt_all[split_index:]
y_test = y_all[split_index:]

print("Train sizes:", len(X_txt_train), len(y_train))  
print("Test sizes:", len(X_txt_test), len(y_test))

Train sizes: 21984 21984
Test sizes: 5497 5497


In [37]:
class LexiconClassifier():
    def __init__(self):
        self.positive_words = set()
        with open(r"C:\Users\151495\Desktop\Sentiment Analysis\positive.txt", encoding = 'utf-8') as iFile:
            for row in iFile:
                self.positive_words.add(row.strip())

        self.negative_words = set()
        with open(r"C:\Users\151495\Desktop\Sentiment Analysis\negative.txt", encoding='iso-8859-1') as iFile:
            for row in iFile:
                self.negative_words.add(row.strip())

    def predict(self, sentence):
        """
            Returns a sentiment prediction give an input string.
            
            Keyword arguments:
            sentence -- string (e.g., "This is good good good")
            
            Returns:
            pred -- a string ("postive, "negative", or "neutral")
        """
        num_pos_words = 0
        num_neg_words = 0
        for word in sentence.lower().split():
            if word in self.positive_words:
                num_pos_words += 1
            elif word in self.negative_words:
                num_neg_words += 1
        
        pred = 'neutral'        
        if num_pos_words > num_neg_words:
            pred = 'positive'
        elif num_pos_words < num_neg_words:
            pred = 'negative'
            
        return pred
    
    def count_pos_words(self, sentence):
        """
            Returns the number of positive words in string
            
            Keyword arguments:
            sentence -- string (e.g., "This is good good good")
            
            Returns:
            pred -- an integer (e.g., 3)
        """
        num_pos_words = 0
        for word in sentence.lower().split():
            if word in self.positive_words:
                num_pos_words += 1
        return num_pos_words

    def count_neg_words(self, sentence):
        """
            Returns the number of negative words in string
            
            Keyword arguments:
            sentence -- string (e.g., "This is good good good")
            
            Returns:
            pred -- an integer (e.g., 3)
        """
        num_neg_words = 0
        for word in sentence.lower().split():
            if word in self.negative_words:
                num_neg_words += 1
        return num_neg_words

In [20]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score


lexicon_model = LexiconClassifier()
lex_test_preds = [] 

# Loop over X_txt_test
#    for each string in X_txt_test (i.e., for each item in the list), pass it to LexiconClassifiers .predict() method
#    append the prediction to lex_test_preds

for text in X_txt_test:
    pred = lexicon_model.predict(text)
    lex_test_preds.append(pred)

precision = precision_score(y_test, lex_test_preds, average="macro", zero_division=0) # Get scores using lex_test_preds and y_test with the precision_score method
recall = recall_score(y_test, lex_test_preds, average="macro", zero_division=0) # Get scores using lex_test_preds and y_test with the recall_score method
f1 = f1_score(y_test, lex_test_preds, average="macro", zero_division=0) # Get scores using lex_test_preds and y_test with the f1_score method

print("Precision: {:.4f}".format(precision))
print("Recall: {:.4f}".format(recall))
print("F1: {:.4f}".format(f1))

Precision: 0.5564
Recall: 0.5505
F1: 0.5440


In [21]:
X_train_lexicon_features = [] 
X_test_lexicon_features = [] 

for text in X_txt_train:
    pos_count = lexicon_model.count_pos_words(text)
    neg_count = lexicon_model.count_neg_words(text)
    X_train_lexicon_features.append([pos_count, neg_count])

for text in X_txt_test:
    pos_count = lexicon_model.count_pos_words(text)
    neg_count = lexicon_model.count_neg_words(text)
    X_test_lexicon_features.append([pos_count, neg_count])

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score

import numpy as np
np.random.seed(42)
import random
random.seed(42)

vectorizer = CountVectorizer(ngram_range=(1, 1))
X_train = vectorizer.fit_transform(X_txt_train)  
X_test = vectorizer.transform(X_txt_test)         

svm = LinearSVC(max_iter=100)
# Create the params with the C values
params = {'C': [0.0001, 0.001, 0.001, 0.01, 0.1, 1, 10, 100]}
# Initialize GridSearchCV
grid_search = GridSearchCV(svm, params, scoring='f1_macro', cv=5)

grid_search.fit(X_train, y_train)
validation_score = grid_search.best_score_ # Get the score from the GridSearchCV "best score"
print("Validation F1: {:.4f}".format(validation_score))

svm_test_predictions = grid_search.predict(X_test) # "predict" on X_test 

precision = precision_score(y_test, svm_test_predictions, average="macro", zero_division=0) # Get scores using svm_test_predictions and y_test with the precision_score method
recall = recall_score(y_test, svm_test_predictions, average="macro", zero_division=0)
f1 = f1_score(y_test, svm_test_predictions, average="macro", zero_division=0)
print("Precision: {:.4f}".format(precision))
print("Recall: {:.4f}".format(recall))
print("F1: {:.4f}".format(f1))

C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn

Validation F1: 0.6876
Precision: 0.7052
Recall: 0.6926
F1: 0.6974


In [25]:
import scipy.sparse as sp
from scipy.sparse import hstack
import numpy as np
np.random.seed(42)
import random
random.seed(42)

X_train_w_lex = vectorizer.transform(X_txt_train) # This will be the matrix from CountVectorizer (X_txt_train)
X_test_w_lex = vectorizer.transform(X_txt_test)

lex_train_matrix = sp.csr_matrix(np.array(X_train_lexicon_features))
lex_test_matrix = sp.csr_matrix(np.array(X_test_lexicon_features))

X_train_w_lex = hstack([X_train_w_lex, lex_train_matrix])
X_test_w_lex = hstack([X_test_w_lex, lex_test_matrix])

svm_lex = LinearSVC(max_iter=100) #iteration can be extend to 100+ but model runs for hours..

params = {'C': [0.0001, 0.001, 0.001, 0.01, 0.1, 1, 10, 100]}

grid_search_lex = GridSearchCV(svm_lex, params, scoring='f1_macro', cv=5)

grid_search_lex.fit(X_train_w_lex, y_train)

validation_score = grid_search_lex.best_score_
print("Validation F1: {:.4f}".format(validation_score))

svm_lex_test_predictions = grid_search_lex.predict(X_test_w_lex) # Get predictions on X_test_w_lex

precision = precision_score(y_test, svm_lex_test_predictions, average="macro", zero_division=0) # Get scores using svm_test_predictions and y_test with the precision_score method
recall = recall_score(y_test, svm_lex_test_predictions, average="macro", zero_division=0)
f1 = f1_score(y_test, svm_lex_test_predictions, average="macro", zero_division=0)
print("Precision: {:.4f}".format(precision))
print("Recall: {:.4f}".format(recall))
print("F1: {:.4f}".format(f1))

C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn

Validation F1: 0.6922
Precision: 0.7099
Recall: 0.6992
F1: 0.7036


C:\Users\151495\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [26]:
num_tweets = 0
for text, svm_pred, svm_lex_pred, lex_pred, y  in zip(X_txt_test, svm_test_predictions, svm_lex_test_predictions, lex_test_preds, y_test):
    print("Tweet: {}".format(text))
    print("Ground-Truth Class: {}".format(y))
    print("SVM Prediction: {}".format(svm_pred))
    print("SVM+Lexicon Prediction: {}".format(svm_lex_pred))
    print("Lexicon Model Prediction: {}".format(lex_pred))
    print()
    
    num_tweets += 1
    if num_tweets == 20:
        break

Tweet:  oh yeah - love his choregoraphy. the pants...not so much.
Ground-Truth Class: neutral
SVM Prediction: neutral
SVM+Lexicon Prediction: positive
Lexicon Model Prediction: positive

Tweet: _JessicaB_**** yip.....aw gonna miss them on bb
Ground-Truth Class: negative
SVM Prediction: negative
SVM+Lexicon Prediction: negative
Lexicon Model Prediction: negative

Tweet: _violence heyyyy babyy
Ground-Truth Class: negative
SVM Prediction: neutral
SVM+Lexicon Prediction: neutral
Lexicon Model Prediction: neutral

Tweet: Up at 6am on Sunday... Going to meet my mom for breakfast at the beach!
Ground-Truth Class: neutral
SVM Prediction: neutral
SVM+Lexicon Prediction: neutral
Lexicon Model Prediction: neutral

Tweet: so the Today show still hasn`t gotten in touch with me, i wish they would so i can take my friends and myself to the NKOTB show
Ground-Truth Class: neutral
SVM Prediction: negative
SVM+Lexicon Prediction: neutral
Lexicon Model Prediction: positive

Tweet: Just checked email and g